# ***Step 1 : Dataset Understanding***

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Load datasets
df_fake = pd.read_csv("/kaggle/input/dataset/Fake.csv")
df_true = pd.read_csv("/kaggle/input/dataset/True.csv")

# Labels
df_fake["label"] = 0
df_true["label"] = 1

# Merge & shuffle
df = pd.concat([df_fake, df_true], ignore_index=True)
df = df.sample(frac=1, random_state=42)

df.info()
df.head()

In [ ]:
# Missing values
df.isnull().sum()

In [ ]:
# Duplicates
df.drop_duplicates(inplace=True)

In [ ]:
# Date processing
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["Year"] = df["date"].dt.year
df["Month"] = df["date"].dt.month_name()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="label", hue="label", data=df, palette="hls", legend=False)
plt.title("Fake vs Real News")
plt.show()

In [ ]:
# Distribution by Year
plt.figure(figsize=(12,6))
sns.countplot(x="Year", data=df, order=df["Year"].value_counts().index)
plt.show()

In [ ]:
# Distribution by Month
plt.figure(figsize=(12,6))
sns.countplot(x="Month", data=df, order=df["Month"].value_counts().index)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Distribution by Subject
plt.figure(figsize=(12,6))
sns.countplot(x="subject", data=df, order=df["subject"].value_counts().index)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Label vs Subject
plt.figure(figsize=(12,6))
sns.countplot(x="subject", hue="label", data=df)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# WordCloud
from wordcloud import WordCloud, STOPWORDS

text = " ".join(df["text"].astype(str))
wc = WordCloud(width=3000, height=2000, stopwords=STOPWORDS).generate(text)
plt.imshow(wc)
plt.axis("off")
plt.show()

# ***Step 2 : Data Preprocessing & Feature Extraction***

In [ ]:
import re
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Nettoyage texte
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return " ".join(tokens)

df["text"] = df["text"].apply(preprocess)

# Séquence + padding pour LSTM
max_words = 20000  # vocab size
max_seq_len = 300  # longueur maximale des séquences

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df["text"])

X = tokenizer.texts_to_sequences(df["text"])
X = pad_sequences(X, maxlen=max_seq_len)

y = df["label"].values

# Split train/validation
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
tokenizer.word_index

# ***Step 4 : Model Training***

***Model LSTM***

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

word_counts = min(max_words, len(tokenizer.word_index)+1)

model = Sequential([
    Embedding(input_dim=word_counts, output_dim=300, input_length=max_seq_len),
    LSTM(256, return_sequences=True),
    LSTM(128, return_sequences=True),
    LSTM(64, return_sequences=False),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=10,
    batch_size=64
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tr_acc = history.history['accuracy']
tr_loss = history.history['loss']
val_acc = history.history['val_accuracy']
val_loss = history.history['val_loss']

index_loss = np.argmin(val_loss)
val_lowest = val_loss[index_loss]
index_acc = np.argmax(val_acc)
acc_highest = val_acc[index_acc]

Epochs = [i+1 for i in range(len(tr_acc))]
loss_label = f'best epoch = {index_loss + 1}'
acc_label  = f'best epoch = {index_acc + 1}'

plt.figure(figsize=(20,8))
plt.style.use('fivethirtyeight')

# Loss
plt.subplot(1,2,1)
plt.plot(Epochs, tr_loss, 'r', label='Training Loss')
plt.plot(Epochs, val_loss, 'g', label='Validation Loss')
plt.scatter(index_loss+1, val_lowest, s=150, c='blue', label=loss_label)
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Accuracy
plt.subplot(1,2,2)
plt.plot(Epochs, tr_acc, 'r', label='Training Accuracy')
plt.plot(Epochs, val_acc, 'g', label='Validation Accuracy')
plt.scatter(index_acc+1, acc_highest, s=150, c='blue', label=acc_label)
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
y_pred_prob = model.predict(X_valid)
y_pred = (y_pred_prob > 0.5).astype(int)

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

acc  = accuracy_score(y_valid, y_pred)
prec = precision_score(y_valid, y_pred)
rec  = recall_score(y_valid, y_pred)
f1   = f1_score(y_valid, y_pred)
roc  = roc_auc_score(y_valid, y_pred_prob)

print("===== LSTM Evaluation =====")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {roc:.4f}")

In [ ]:
cm = confusion_matrix(y_valid, y_pred)
print("Confusion Matrix:")
print(cm)

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
test_df = pd.read_csv("/kaggle/input/ttestmd/test.csv", sep=';')
print(test_df.columns)
test_df["text"] = test_df["text"].apply(preprocess)
X_test_kaggle = tokenizer.texts_to_sequences(test_df["text"])
X_test_kaggle = pad_sequences(X_test_kaggle, maxlen=max_seq_len)

kaggle_preds = (model.predict(X_test_kaggle) > 0.5).astype(int)

submission = pd.DataFrame({
    "id": test_df.index,
    "label": kaggle_preds.flatten()
})
submission.to_csv("submission_lstm.csv", index=False)

***Model CNN***

In [ ]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalMaxPooling1D

# ===== CNN Model =====

model_cnn = Sequential([
    Embedding(input_dim=word_counts, output_dim=300),

    Conv1D(filters=256, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),

    Conv1D(filters=128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),

    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_cnn.summary()

history_cnn = model_cnn.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=10,
    batch_size=64
)


In [ ]:
tr_acc = history_cnn.history['accuracy']
tr_loss = history_cnn.history['loss']
val_acc = history_cnn.history['val_accuracy']
val_loss = history_cnn.history['val_loss']

Epochs = [i+1 for i in range(len(tr_acc))]

plt.figure(figsize=(20,8))

# Loss
plt.subplot(1,2,1)
plt.plot(Epochs, tr_loss, label='Training Loss')
plt.plot(Epochs, val_loss, label='Validation Loss')
plt.title('CNN - Loss')
plt.legend()

# Accuracy
plt.subplot(1,2,2)
plt.plot(Epochs, tr_acc, label='Training Accuracy')
plt.plot(Epochs, val_acc, label='Validation Accuracy')
plt.title('CNN - Accuracy')
plt.legend()

plt.show()

In [ ]:
y_pred_prob_cnn = model_cnn.predict(X_valid)
y_pred_cnn = (y_pred_prob_cnn > 0.5).astype(int)

acc  = accuracy_score(y_valid, y_pred_cnn)
prec = precision_score(y_valid, y_pred_cnn)
rec  = recall_score(y_valid, y_pred_cnn)
f1   = f1_score(y_valid, y_pred_cnn)
roc  = roc_auc_score(y_valid, y_pred_prob_cnn)

print("===== CNN Evaluation =====")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {roc:.4f}")

In [ ]:
cm_cnn = confusion_matrix(y_valid, y_pred_cnn)

print("Confusion Matrix CNN:")
print(cm_cnn)

plt.figure(figsize=(6,5))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("CNN Confusion Matrix")
plt.show()


In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Charger test set
test_df = pd.read_csv("/kaggle/input/ttestmd/test.csv", sep=';')

# Preprocessing texte
test_df["text"] = test_df["text"].apply(preprocess)

# Tokenizer + padding
X_test_kaggle = tokenizer.texts_to_sequences(test_df["text"])
X_test_kaggle = pad_sequences(X_test_kaggle, maxlen=max_seq_len)

# Prediction avec CNN
kaggle_preds_cnn = (model_cnn.predict(X_test_kaggle) > 0.5).astype(int)

# Créer le DataFrame submission
submission_cnn = pd.DataFrame({
    "id": test_df.index,
    "label": kaggle_preds_cnn.flatten()
})

# Sauvegarder CSV
submission_cnn.to_csv("submission_cnn.csv", index=False)

print("submission_cnn.csv created successfully ✅")
submission_cnn.head()
